# 03E — M3 cross-language validation and final selection

This notebook integrates the independent Python and Julia M3 searches.

It reads:

- `pdata/m3_python_best_candidates.csv` from `03C_M3_python.ipynb`;
- `pdata/revision2026_julia_j2_benchmark.csv` from `03D_M3_julia.ipynb`;
- `pdata/revision2026_julia_j2_mixture_diagnostics.csv` from `03D_M3_julia.ipynb`;
- `pdata/table_iii_environmental_model_comparison.csv` from `03B_environmental_models.ipynb`.

The notebook then:

1. standardizes the Python and Julia M3 parameterizations to the reporting convention *mu1 < mu2*;
2. evaluates **both language candidates with the canonical Python physical M3 likelihood**;
3. checks that the Julia-reported likelihood agrees with the independent Python evaluation;
4. selects the lowest cross-evaluated best-found candidate for each class;
5. reconstructs manuscript **Table V** (M0 / M1 / M3);
6. reconstructs manuscript **Table IX** (selected M3 parameters);
7. exports the final cross-language audit tables to `pdata/`.

## Interpretation

Finite-mixture likelihoods can contain nearby local modes. The selected M3 estimates are therefore described as **cross-language validated best-found candidates**, not guaranteed global optima.

For ALL and SG, nearby mixture modes are expected. For IG, one component is nearly degenerate, so parameter representatives can differ numerically while giving essentially the same likelihood.


## 1. Environment, canonical likelihood, and real data

This final cross-language audit is defined for the authorized annual dataset used in the manuscript. It does not silently substitute the public synthetic dataset.

During the current notebook-splitting stage, the canonical Python M3 likelihood remains in `revision_models.py`. The later AntiGravity refactor can move shared functions into `modules/` after all numerical outputs are frozen.


In [2]:
from pathlib import Path
import math

import numpy as np
import pandas as pd

from IPython.display import display

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
PDATA_DIR = ROOT / "pdata"

REAL_DATA = DATA_DIR / "M_1920_2023.csv"

if not REAL_DATA.exists():
    raise FileNotFoundError(
        "03E is the canonical real-data cross-language audit and requires "
        "data/M_1920_2023.csv."
    )

try:
    import revision_models as rm
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "03E currently requires the canonical revision_models.py together "
        "with its current dependencies (models.py and analysis_helpers.py) "
        "in the project root. These will be modularized later."
    ) from exc


df = pd.read_csv(REAL_DATA)

EXPECTED_COLUMNS = [
    "Year",
    "ALL", "D_ALL",
    "SG", "D_SG",
    "IG", "D_IG",
]

missing = [c for c in EXPECTED_COLUMNS if c not in df.columns]

if missing:
    raise RuntimeError(
        f"Annual data have an unexpected schema. Missing columns: {missing}"
    )

for c in EXPECTED_COLUMNS:
    df[c] = pd.to_numeric(df[c], errors="raise")

print(f"Data: {REAL_DATA}")
print(f"Rows: {len(df)} | Years: {int(df['Year'].min())}–{int(df['Year'].max())}")


Data: ./data/M_1920_2023.csv
Rows: 104 | Years: 1920–2023


## 2. Load 03B, 03C, and 03D outputs

The notebook fails early if any upstream result is missing or uses an unexpected schema.


In [3]:
P03B = PDATA_DIR / "table_iii_environmental_model_comparison.csv"
P03C = PDATA_DIR / "m3_python_best_candidates.csv"
P03D_BENCH = PDATA_DIR / "revision2026_julia_j2_benchmark.csv"
P03D_DIAG = PDATA_DIR / "revision2026_julia_j2_mixture_diagnostics.csv"

for p in [P03B, P03C, P03D_BENCH, P03D_DIAG]:
    if not p.exists():
        raise FileNotFoundError(f"Missing upstream output: {p}")


df_03b = pd.read_csv(P03B)
df_py = pd.read_csv(P03C)
df_jl = pd.read_csv(P03D_BENCH)
df_jl_diag = pd.read_csv(P03D_DIAG)

required_03b = {
    "class", "model", "T", "k", "NLL", "AIC", "BIC",
    "boundary", "solution",
}
required_py = {
    "class", "source", "NLL", "mu1", "sigma1", "mu2", "sigma2", "pi",
}
required_jl = {
    "class", "model", "GH_N", "optimizer", "start_count",
    "successful_starts", "best_nll", "mu1_hat", "mu2_hat",
    "sigma1_hat", "sigma2_hat", "pi_hat",
}
required_jl_diag = {
    "class", "final_nll",
}

checks = [
    ("03B", df_03b, required_03b),
    ("03C", df_py, required_py),
    ("03D benchmark", df_jl, required_jl),
    ("03D diagnostics", df_jl_diag, required_jl_diag),
]

for label, frame, required in checks:
    missing = required - set(frame.columns)
    if missing:
        raise RuntimeError(
            f"{label} output has an unexpected schema. Missing: {sorted(missing)}"
        )

CLASSES = ["ALL", "SG", "IG"]

assert set(df_py["class"].astype(str)) == set(CLASSES)
assert set(df_jl["class"].astype(str)) == set(CLASSES)
assert len(df_py) == 3
assert len(df_jl) == 3

assert np.all(pd.to_numeric(df_jl["GH_N"]) == 100)
assert np.all(df_jl["optimizer"].astype(str) == "NelderMead")
assert np.all(pd.to_numeric(df_jl["start_count"]) == 180)
assert np.all(pd.to_numeric(df_jl["successful_starts"]) == 180)

print("Upstream file/schema audit: PASS")


Upstream file/schema audit: PASS


## 3. Canonical helper functions

Mixture labels are arbitrary. Every candidate is rewritten to the reporting convention *mu1 < mu2*. When components are swapped, *pi* is replaced by *1-pi*.


In [4]:
def make_grouped_nh(series_n, series_h):
    tmp = pd.DataFrame({
        "n": np.asarray(series_n, dtype=int),
        "h": np.asarray(series_h, dtype=int),
    })

    grouped = (
        tmp.value_counts(["n", "h"])
        .rename("count")
        .reset_index()
        .sort_values(["n", "h"])
        .reset_index(drop=True)
    )

    return (
        grouped["n"].to_numpy(dtype=int),
        grouped["h"].to_numpy(dtype=int),
        grouped["count"].to_numpy(dtype=int),
    )


def order_components(mu1, sigma1, mu2, sigma2, pi):
    mu1 = float(mu1)
    sigma1 = float(sigma1)
    mu2 = float(mu2)
    sigma2 = float(sigma2)
    pi = float(pi)

    if mu1 <= mu2:
        return {
            "mu1": mu1,
            "sigma1": sigma1,
            "mu2": mu2,
            "sigma2": sigma2,
            "pi": pi,
            "components_swapped": False,
        }

    return {
        "mu1": mu2,
        "sigma1": sigma2,
        "mu2": mu1,
        "sigma2": sigma1,
        "pi": 1.0 - pi,
        "components_swapped": True,
    }


def evaluate_m3_python(cls, params, gh_n=100):
    n_arr, h_arr, c_arr = make_grouped_nh(
        df[cls],
        df["D_" + cls],
    )

    return float(
        rm.nll_probit_normal_mixture_physical(
            mu1=float(params["mu1"]),
            mu2=float(params["mu2"]),
            sigma1=float(params["sigma1"]),
            sigma2=float(params["sigma2"]),
            pi=float(params["pi"]),
            n_arr=n_arr,
            h_arr=h_arr,
            c_arr=c_arr,
            GH_N=gh_n,
        )
    )


## 4. Put the Python and Julia candidates into one schema

The Julia benchmark uses `best_nll`, `mu1_hat`, ... whereas 03C uses `NLL`, `mu1`, ... . This cell standardizes both without modifying their reported values.


In [5]:
candidate_rows = []

for cls in CLASSES:
    py = df_py.loc[df_py["class"] == cls].iloc[0]
    py_ordered = order_components(
        py["mu1"], py["sigma1"],
        py["mu2"], py["sigma2"],
        py["pi"],
    )

    candidate_rows.append({
        "class": cls,
        "source": "Python",
        "reported_NLL": float(py["NLL"]),
        **py_ordered,
    })

    jl = df_jl.loc[df_jl["class"] == cls].iloc[0]
    jl_ordered = order_components(
        jl["mu1_hat"], jl["sigma1_hat"],
        jl["mu2_hat"], jl["sigma2_hat"],
        jl["pi_hat"],
    )

    candidate_rows.append({
        "class": cls,
        "source": "Julia",
        "reported_NLL": float(jl["best_nll"]),
        **jl_ordered,
    })


df_candidates = pd.DataFrame(candidate_rows)

assert (df_candidates["mu1"] <= df_candidates["mu2"]).all()

display(df_candidates.round(10))


,class,source,reported_NLL,mu1,sigma1,mu2,sigma2,pi,components_swapped
0,ALL,Python,430.271292,-2.580053,0.697085,-2.411760,3.766324e-01,0.199835,False
1,ALL,Julia,430.271292,-2.580053,0.697083,-2.411763,3.766339e-01,0.199837,False
2,SG,Python,416.263555,-3.195875,1.079321,-1.968803,3.424642e-01,0.124965,False
3,SG,Julia,416.177207,-2.837232,0.895836,-1.962688,3.374465e-01,0.161096,False
4,IG,Python,180.112948,-3.419428,0.499547,-2.608762,5.140000e-08,0.883577,False
5,IG,Julia,180.112948,-3.419425,0.499542,-2.608766,3.282300e-05,0.883578,False


## 5. Python cross-evaluation of both language candidates

This is the key cross-language check in the split workflow.

For each Python and Julia candidate, the physical parameter vector is evaluated with the canonical Python M3 likelihood using the same 100-node Gauss–Hermite rule.

A Julia candidate is accepted only if its Julia-reported NLL agrees with the independent Python evaluation within `1e-6`.


In [6]:
CROSS_EVAL_TOL = 1e-6

cross_rows = []

for _, r in df_candidates.iterrows():
    params = {
        "mu1": r["mu1"],
        "sigma1": r["sigma1"],
        "mu2": r["mu2"],
        "sigma2": r["sigma2"],
        "pi": r["pi"],
    }

    nll_py_eval = evaluate_m3_python(
        str(r["class"]),
        params,
        gh_n=100,
    )

    delta = nll_py_eval - float(r["reported_NLL"])

    cross_rows.append({
        **r.to_dict(),
        "python_cross_eval_NLL": nll_py_eval,
        "cross_eval_delta": delta,
        "cross_eval_abs_delta": abs(delta),
        "cross_eval_pass": abs(delta) <= CROSS_EVAL_TOL,
    })


df_cross = pd.DataFrame(cross_rows)

display(
    df_cross[
        [
            "class", "source", "reported_NLL",
            "python_cross_eval_NLL", "cross_eval_delta",
            "cross_eval_pass",
        ]
    ].round(12)
)

if not bool(df_cross["cross_eval_pass"].all()):
    bad = df_cross.loc[~df_cross["cross_eval_pass"]]
    raise RuntimeError(
        "At least one M3 candidate failed the Python cross-evaluation tolerance.\n"
        + bad.to_string(index=False)
    )

print("Cross-language physical-likelihood audit: PASS")


,class,source,reported_NLL,python_cross_eval_NLL,cross_eval_delta,cross_eval_pass
0,ALL,Python,430.271292,430.271292,-0.000000e+00,True
1,ALL,Julia,430.271292,430.271292,-2.300000e-11,True
2,SG,Python,416.263555,416.263555,-0.000000e+00,True
3,SG,Julia,416.177207,416.177207,8.000000e-12,True
4,IG,Python,180.112948,180.112948,-0.000000e+00,True
5,IG,Julia,180.112948,180.112948,3.000000e-12,True


Cross-language physical-likelihood audit: PASS


## 6. Julia multi-start mode audit

The Julia full search used 180 starts for each class. We retain the nearest distinct likelihood mode for context, because ALL and SG have nearby modes and IG has a nearly degenerate component.


In [7]:
mode_rows = []

for cls in CLASSES:
    sub = df_jl_diag.loc[df_jl_diag["class"] == cls].copy()
    sub["final_nll"] = pd.to_numeric(sub["final_nll"], errors="coerce")
    sub = (
        sub.loc[np.isfinite(sub["final_nll"])]
        .sort_values("final_nll")
        .reset_index(drop=True)
    )

    if len(sub) != 180:
        raise RuntimeError(
            f"{cls}: expected 180 finite Julia starts; found {len(sub)}."
        )

    best = float(sub.iloc[0]["final_nll"])
    distinct = sub.loc[np.abs(sub["final_nll"] - best) > 1e-4]
    second = float(distinct.iloc[0]["final_nll"]) if len(distinct) else np.nan

    mode_rows.append({
        "class": cls,
        "best_NLL": best,
        "second_mode_NLL": second,
        "delta_second_vs_best": second - best if np.isfinite(second) else np.nan,
    })


df_modes = pd.DataFrame(mode_rows)
display(df_modes.round(10))


,class,best_NLL,second_mode_NLL,delta_second_vs_best
0,ALL,430.271292,430.284896,0.013603
1,SG,416.177207,416.268225,0.091017
2,IG,180.112948,180.376383,0.263435


## 7. Select the cross-language validated best-found candidate

Selection is performed **after** cross-evaluation. For each class, the candidate with the lower canonical Python cross-evaluated NLL is retained.

This is a best-found selection across the two independent searches, not a proof of global optimality.


In [8]:
selected_rows = []
selection_compare_rows = []

for cls in CLASSES:
    sub = (
        df_cross.loc[df_cross["class"] == cls]
        .sort_values("python_cross_eval_NLL")
        .reset_index(drop=True)
    )

    if len(sub) != 2:
        raise RuntimeError(f"{cls}: expected exactly two language candidates.")

    best = sub.iloc[0]
    other = sub.iloc[1]

    selected_rows.append({
        "class": cls,
        "selected_source": str(best["source"]),
        "NLL": float(best["python_cross_eval_NLL"]),
        "reported_NLL": float(best["reported_NLL"]),
        "mu1": float(best["mu1"]),
        "sigma1": float(best["sigma1"]),
        "mu2": float(best["mu2"]),
        "sigma2": float(best["sigma2"]),
        "pi": float(best["pi"]),
        "validation_status": "PASS",
        "selection_label": "cross-language validated best-found candidate",
    })

    selection_compare_rows.append({
        "class": cls,
        "best_source": str(best["source"]),
        "other_source": str(other["source"]),
        "best_NLL": float(best["python_cross_eval_NLL"]),
        "other_NLL": float(other["python_cross_eval_NLL"]),
        "NLL_gap_other_minus_best": float(
            other["python_cross_eval_NLL"] - best["python_cross_eval_NLL"]
        ),
    })


df_selected = pd.DataFrame(selected_rows)
df_selection_compare = pd.DataFrame(selection_compare_rows)

print("=== Selected cross-language M3 candidates ===")
display(df_selected.round(12))

print("=== Python-vs-Julia selection gaps ===")
display(df_selection_compare.round(12))


=== Selected cross-language M3 candidates ===


,class,selected_source,NLL,reported_NLL,mu1,sigma1,mu2,sigma2,pi,validation_status,selection_label
0,ALL,Julia,430.271292,430.271292,-2.580053,0.697083,-2.411763,3.766339e-01,0.199837,PASS,cross-language validated best-found candidate
1,SG,Julia,416.177207,416.177207,-2.837232,0.895836,-1.962688,3.374465e-01,0.161096,PASS,cross-language validated best-found candidate
2,IG,Python,180.112948,180.112948,-3.419428,0.499547,-2.608762,5.140100e-08,0.883577,PASS,cross-language validated best-found candidate


=== Python-vs-Julia selection gaps ===


,class,best_source,other_source,best_NLL,other_NLL,NLL_gap_other_minus_best
0,ALL,Julia,Python,430.271292,430.271292,5.861000e-09
1,SG,Julia,Python,416.177207,416.263555,8.634788e-02
2,IG,Python,Julia,180.112948,180.112948,3.026800e-08


## 8. Reconstruct manuscript Table V

Table V compares M0, M1, and the final selected M3. M0 and M1 are taken directly from the completed 03B output; M3 is reconstructed from the cross-language selected candidate with nominal parameter count *k=5*.


In [9]:
TABLE_V_MODELS = ["M0", "M1", "M3"]

table_v_rows = []

for cls in CLASSES:
    base = df_03b.loc[
        (df_03b["class"] == cls)
        & (df_03b["model"].isin(["M0", "M1"]))
    ].copy()

    if set(base["model"]) != {"M0", "M1"}:
        raise RuntimeError(f"{cls}: 03B does not contain exactly M0 and M1 rows.")

    for model in ["M0", "M1"]:
        r = base.loc[base["model"] == model].iloc[0]
        table_v_rows.append({
            "class": cls,
            "model": model,
            "k": int(r["k"]),
            "NLL": float(r["NLL"]),
            "AIC": float(r["AIC"]),
            "BIC": float(r["BIC"]),
            "source": "03B",
        })

    s = df_selected.loc[df_selected["class"] == cls].iloc[0]
    T = int(base.iloc[0]["T"])
    k = 5
    nll = float(s["NLL"])

    table_v_rows.append({
        "class": cls,
        "model": "M3",
        "k": k,
        "NLL": nll,
        "AIC": 2.0 * nll + 2.0 * k,
        "BIC": 2.0 * nll + k * math.log(T),
        "source": str(s["selected_source"]),
    })


table_v = pd.DataFrame(table_v_rows)

table_v["class"] = pd.Categorical(
    table_v["class"], categories=CLASSES, ordered=True
)
table_v["model"] = pd.Categorical(
    table_v["model"], categories=TABLE_V_MODELS, ordered=True
)
table_v = table_v.sort_values(["class", "model"]).reset_index(drop=True)

print("=== Table V: M0 / M1 / M3 ===")
display(
    table_v.assign(
        NLL=table_v["NLL"].round(3),
        AIC=table_v["AIC"].round(3),
        BIC=table_v["BIC"].round(3),
    )
)


=== Table V: M0 / M1 / M3 ===


,class,model,k,NLL,AIC,BIC,source
0,ALL,M0,2,431.364,866.727,872.016,03B
1,ALL,M1,3,413.825,833.651,841.584,03B
2,ALL,M3,5,430.271,870.543,883.765,Julia
3,SG,M0,2,417.709,839.417,844.706,03B
4,SG,M1,3,401.342,808.684,816.618,03B
5,SG,M3,5,416.177,842.354,855.576,Julia
6,IG,M0,2,182.382,368.763,374.052,03B
7,IG,M1,3,180.781,367.563,375.496,03B
8,IG,M3,5,180.113,370.226,383.448,Python


## 9. Reconstruct manuscript Table IX

The component means are reported in the convention *mu1 < mu2*. The IG value of *sigma2* is expected to be near zero.


In [10]:
table_ix = df_selected[
    [
        "class",
        "mu1", "sigma1",
        "mu2", "sigma2",
        "pi",
        "selected_source",
    ]
].copy()

print("=== Table IX: cross-language validated best-found M3 parameters ===")
display(table_ix.round(10))


=== Table IX: cross-language validated best-found M3 parameters ===


,class,mu1,sigma1,mu2,sigma2,pi,selected_source
0,ALL,-2.580053,0.697083,-2.411763,3.766339e-01,0.199837,Julia
1,SG,-2.837232,0.895836,-1.962688,3.374465e-01,0.161096,Julia
2,IG,-3.419428,0.499547,-2.608762,5.140000e-08,0.883577,Python


## 10. V3 manuscript regression audit

The current manuscript reports Table V values rounded to three decimals and Table IX parameters with the displayed precision below.

This cell is an audit only. It never replaces computed results with manuscript constants.


In [11]:
# ------------------------------------------------------------
# Table V audit: displayed values in V3
# ------------------------------------------------------------

V3_TABLE_V = pd.DataFrame([
    ["ALL", "M0", 2, 431.364, 866.727, 872.016],
    ["ALL", "M1", 3, 413.825, 833.651, 841.584],
    ["ALL", "M3", 5, 430.271, 870.543, 883.765],
    ["SG",  "M0", 2, 417.709, 839.417, 844.706],
    ["SG",  "M1", 3, 401.342, 808.684, 816.618],
    ["SG",  "M3", 5, 416.177, 842.354, 855.576],
    ["IG",  "M0", 2, 182.382, 368.763, 374.052],
    ["IG",  "M1", 3, 180.781, 367.563, 375.496],
    ["IG",  "M3", 5, 180.113, 370.226, 383.448],
], columns=["class", "model", "k", "NLL", "AIC", "BIC"])

observed_v = table_v[["class", "model", "k", "NLL", "AIC", "BIC"]].copy()
observed_v["class"] = observed_v["class"].astype(str)
observed_v["model"] = observed_v["model"].astype(str)

merged_v = observed_v.merge(
    V3_TABLE_V,
    on=["class", "model", "k"],
    suffixes=("_obs", "_ref"),
)

for c in ["NLL", "AIC", "BIC"]:
    merged_v[f"rounded_delta_{c}"] = (
        merged_v[f"{c}_obs"].round(3) - merged_v[f"{c}_ref"]
    )

max_table_v_delta = float(
    np.max(
        np.abs(
            merged_v[[
                "rounded_delta_NLL",
                "rounded_delta_AIC",
                "rounded_delta_BIC",
            ]].to_numpy()
        )
    )
)

print(f"Maximum rounded Table V difference: {max_table_v_delta:.6g}")
assert max_table_v_delta < 5e-4
print("Table V audit: PASS")

# ------------------------------------------------------------
# Table IX audit: displayed V3 values
# ------------------------------------------------------------

V3_TABLE_IX = pd.DataFrame([
    ["ALL", -2.580053, 0.697083, -2.411763, 0.376634, 0.199837],
    ["SG",  -2.837232, 0.895836, -1.962688, 0.337446, 0.161096],
    ["IG",  -3.419428, 0.499547, -2.608762, 5.14e-8, 0.883577],
], columns=["class", "mu1", "sigma1", "mu2", "sigma2", "pi"])

obs_ix = table_ix[["class", "mu1", "sigma1", "mu2", "sigma2", "pi"]].copy()
merged_ix = obs_ix.merge(V3_TABLE_IX, on="class", suffixes=("_obs", "_ref"))

# Six displayed decimals for ordinary parameters; IG sigma2 is compared directly.
for c in ["mu1", "sigma1", "mu2", "pi"]:
    merged_ix[f"rounded_match_{c}"] = (
        merged_ix[f"{c}_obs"].round(6)
        == merged_ix[f"{c}_ref"].round(6)
    )

sigma2_ok = []
for _, r in merged_ix.iterrows():
    if r["class"] == "IG":
        sigma2_ok.append(
            abs(float(r["sigma2_obs"]) - float(r["sigma2_ref"])) < 5e-9
        )
    else:
        sigma2_ok.append(
            round(float(r["sigma2_obs"]), 6) == round(float(r["sigma2_ref"]), 6)
        )

merged_ix["sigma2_match"] = sigma2_ok

match_cols = [
    "rounded_match_mu1",
    "rounded_match_sigma1",
    "rounded_match_mu2",
    "rounded_match_pi",
    "sigma2_match",
]

print("\nTable IX display-precision audit:")
display(merged_ix[["class"] + match_cols])

assert bool(merged_ix[match_cols].all(axis=1).all())
print("Table IX audit: PASS")


Maximum rounded Table V difference: 0
Table V audit: PASS

Table IX display-precision audit:


,class,rounded_match_mu1,rounded_match_sigma1,rounded_match_mu2,rounded_match_pi,sigma2_match
0,ALL,True,True,True,True,True
1,SG,True,True,True,True,True
2,IG,True,True,True,True,True


Table IX audit: PASS


## 11. Scientific consistency checks

These checks encode only claims already used in the manuscript comparison:

- M3 improves NLL over M0 in every class;
- for ALL and SG, the M1 likelihood gain over M0 is much larger than the M3 gain;
- the finite-mixture solution is never labeled a guaranteed global optimum.


In [12]:
scientific_rows = []

for cls in CLASSES:
    sub = table_v.loc[table_v["class"].astype(str) == cls]

    nll0 = float(sub.loc[sub["model"].astype(str) == "M0", "NLL"].iloc[0])
    nll1 = float(sub.loc[sub["model"].astype(str) == "M1", "NLL"].iloc[0])
    nll3 = float(sub.loc[sub["model"].astype(str) == "M3", "NLL"].iloc[0])

    gain_m1 = nll0 - nll1
    gain_m3 = nll0 - nll3

    assert gain_m3 > 0.0

    if cls in {"ALL", "SG"}:
        assert gain_m1 > gain_m3

    scientific_rows.append({
        "class": cls,
        "M1_gain_vs_M0": gain_m1,
        "M3_gain_vs_M0": gain_m3,
        "M1_gain_exceeds_M3_gain": gain_m1 > gain_m3,
    })


df_science = pd.DataFrame(scientific_rows)
display(df_science.round(8))

print(
    "\nInterpretation: M3 is a cross-language validated best-found "
    "flexible environmental-null candidate; no global-optimum claim is made."
)


,class,M1_gain_vs_M0,M3_gain_vs_M0,M1_gain_exceeds_M3_gain
0,ALL,17.538404,1.092418,True
1,SG,16.366364,1.531327,True
2,IG,1.600412,2.268738,False



Interpretation: M3 is a cross-language validated best-found flexible environmental-null candidate; no global-optimum claim is made.


## 12. Export final cross-language results

The split workflow uses descriptive filenames. Two compatibility files are also written so that downstream code can consume a single final M3 result if needed.


In [13]:
# Full candidate-level cross-language audit.
df_cross.to_csv(
    PDATA_DIR / "m3_cross_language_validation.csv",
    index=False,
)

# One selected final M3 candidate per class.
df_selected.to_csv(
    PDATA_DIR / "m3_cross_language_selected_candidates.csv",
    index=False,
)

# Manuscript tables.
table_v.to_csv(
    PDATA_DIR / "table_v_m0_m1_m3.csv",
    index=False,
)

table_ix.to_csv(
    PDATA_DIR / "table_ix_m3_parameters.csv",
    index=False,
)

# Julia local-mode context.
df_modes.to_csv(
    PDATA_DIR / "m3_julia_mode_audit.csv",
    index=False,
)

# Selection gaps / provenance.
df_selection_compare.to_csv(
    PDATA_DIR / "m3_cross_language_selection_gaps.csv",
    index=False,
)

# Compatibility aliases for the canonical pre-split workflow.
table_ix_compat = table_ix.rename(columns={"selected_source": "source"}).copy()
table_ix_compat.to_csv(
    PDATA_DIR / "revision2026_m3_parameter_estimates_final.csv",
    index=False,
)

print("Saved final 03E outputs:")
for name in [
    "m3_cross_language_validation.csv",
    "m3_cross_language_selected_candidates.csv",
    "m3_cross_language_selection_gaps.csv",
    "m3_julia_mode_audit.csv",
    "table_v_m0_m1_m3.csv",
    "table_ix_m3_parameters.csv",
    "revision2026_m3_parameter_estimates_final.csv",
]:
    print("  ", PDATA_DIR / name)


Saved final 03E outputs:
   ./pdata/m3_cross_language_validation.csv
   ./pdata/m3_cross_language_selected_candidates.csv
   ./pdata/m3_cross_language_selection_gaps.csv
   ./pdata/m3_julia_mode_audit.csv
   ./pdata/table_v_m0_m1_m3.csv
   ./pdata/table_ix_m3_parameters.csv
   ./pdata/revision2026_m3_parameter_estimates_final.csv


## Output summary

A successful run should reproduce the current manuscript values:

### Table V M3 NLL

- ALL: approximately 430.271
- SG: approximately 416.177
- IG: approximately 180.113

### Table IX

- ALL: the selected candidate is expected to agree with the reported six-decimal parameter values;
- SG: the Julia search supplies the lower mode discovered by the full 540-start audit;
- IG: the selected candidate contains a nearly degenerate component with *sigma2* approximately *5.14e-8*.

The notebook intentionally keeps optimization and integration separate:

- 03C = Python search;
- 03D = Julia search;
- 03E = cross-language physical-likelihood validation and final best-found selection.
